## About:

### This notebook evaluates the answers generated from OLMO models and OLMO Models with RAG using as LLM (Gemini here) as Judge. The 3 parameters used for evaluation are : Helpfulness, Relevance and Groundedness. Gemini evaluates the answers and gives a score between 0 to 1 where 0 means worst and 1 means best.

In [1]:
!pip install google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 737.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 650.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [google-generativeai]ogle-ai-generativelanguage]


In [2]:
import pandas as pd
import google.generativeai as genai
import json
import time

/Users/baisakhisarkar/Downloads/OPT_UW_Temp/OLMO1_OLMO2_Data/olmo-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configure the API key
GOOGLE_API_KEY = "Your API Key"
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash", generation_config={"max_output_tokens": 500})



In [ ]:
df = pd.read_csv("olmo_generation_output_RAG.csv", encoding='ISO-8859-1')

model_col = "RAG_OLMo-2-0425-1B-Instruct"  
df = df[['question', 'true_answer', model_col]].dropna()
df = df.rename(columns={model_col: 'generated_answer'})
df.reset_index(drop=True, inplace=True)

sample_df = df.head(5).copy()
sample_df

,question,true_answer,generated_answer
0,"Hi, \nIâm following this tutorial: The LSST...",Quick comment on the code: \n \n \n \n petarz...,You are an astrophysics expert. Please answer ...
1,I have the following C++ class : \n class CcdI...,After several iteration with @ktl and @rowe...,You are an astrophysics expert. Please answer ...
2,Question on how forced photometry will be run ...,I take this to mean that a DIASource which is ...,You are an astrophysics expert. Please answer ...
3,"Hi there, \n Is there some way I find out what...",Hi James \nmaybe \n dafButler.Butler.get_known...,You are an astrophysics expert. Please answer ...
4,Iâm having trouble building FFTW with texinf...,This has now been fixed and 3.3.4 is the curre...,You are an astrophysics expert. Please answer ...


In [ ]:
#Prompt

# def build_prompt(row):
#     return f"""
# You are an expert evaluator for AI-generated answers. Your task is to rate how good the generated answer is based on the question and the reference (ground-truth) answer.

# Please return your ratings in the following format (with values between 0.000 and 1.000, rounded to 3 decimals). Use realistic scores — 0.000 means extremely poor or entirely incorrect, and 1.000 means perfect — both are rare in real-world answers.

# Evaluate the following:

# Question:
# {row['question']}

# Generated Answer:
# {row['generated_answer']}

# Reference Answer:
# {row['true_answer']}

# Rate the generated answer on a scale of 0 to 1 for the following:

# 1. Helpfulness - Does the answer clearly and usefully address the question?
# 2. Relevance - Is the content relevant to the question asked?
# 3. Groundedness - Does the answer stay true to the reference answer?

# Return your response in JSON like:
# {{
#   "helpfulness": 0.xx,
#   "relevance": 0.xx,
#   "groundedness": 0.xx
# }}
# """




def build_prompt(row):
    return f"""
You are a precise evaluator of AI-generated answers. Do not add any explanation or formatting.

Evaluate the generated answer below for the given question and reference answer. Then respond with a JSON object using floating-point numbers rounded to 3 decimal places (e.g., 0.612, 0.876) where 0.000 is the least and 1.000 is the max.
Please return your ratings in the following format (with values between 0.000 and 1.000, rounded to 3 decimals). Use realistic scores — 0.000 means extremely poor or entirely incorrect, and 1.000 means perfect — both are rare in real-world answers.

Question:
{row['question']}

Generated Answer:
{row['generated_answer']}

Reference Answer:
{row['true_answer']}

Rate the generated answer from 0.000 to 1.000 for the following metrics:

# 1. Helpfulness - Does the answer clearly and usefully address the question?
# 2. Relevance - Is the content relevant to the question asked?
# 3. Groundedness - Does the answer stay true to the reference answer?


Return ONLY a raw JSON object like this:
{{
  "helpfulness": 0.xxx,
  "relevance": 0.xxx,
  "groundedness": 0.xxx
}}
"""



In [40]:
import re

results = []

for idx, row in sample_df.iterrows():
    prompt = build_prompt(row)
    try:
        response = model.generate_content(prompt)
        reply = response.text.strip()

        # Extract JSON if Gemini returns it inside a markdown code block
        json_match = re.search(r'\{.*\}', reply, re.DOTALL)
        if json_match:
            parsed_json = json.loads(json_match.group())
        else:
            raise ValueError("No JSON found in Gemini response.")

        results.append({
            "helpfulness": parsed_json.get("helpfulness"),
            "relevance": parsed_json.get("relevance"),
            "groundedness": parsed_json.get("groundedness")
        })
        # results.append({
        #     "helpfulness": round(parsed_json.get("helpfulness", 0), 3),
        #     "relevance": round(parsed_json.get("relevance", 0), 3),
        #     "groundedness": round(parsed_json.get("groundedness", 0), 3)
        # })

    except Exception as e:
        print(f"Error at row {idx}: {e}")
        results.append({
            "helpfulness": None,
            "relevance": None,
            "groundedness": None
        })
    time.sleep(1)



In [42]:

sample_df['helpfulness'] = [r['helpfulness'] for r in results]
sample_df['relevance'] = [r['relevance'] for r in results]
sample_df['groundedness'] = [r['groundedness'] for r in results]

sample_df.to_csv("gemini_scored_answers.csv", index=False)

print(sample_df[['question', 'helpfulness', 'relevance', 'groundedness']])

                                            question  helpfulness  relevance  \
0  Hi, \nIâm following this tutorial:  The LSST...          0.7        0.8   
1  I have the following C++ class : \n class CcdI...          0.0        0.1   
2  Question on how forced photometry will be run ...          0.6        0.8   
3  Hi there, \n Is there some way I find out what...          0.1        0.2   
4  Iâm having trouble building FFTW with texinf...          0.0        0.0   

   groundedness  
0           0.1  
1           0.0  
2           0.4  
3           0.0  
4           0.0  
